# Predicción de mora temprana en microcréditos MYPE
**Notebook 02 · Calidad y preparación de datos**

Entrada: `data/raw/creditos_mype.csv` · Salida: `data/processed/creditos_limpio.csv`

## 4. Calidad y preparación de datos

Se toma como referencia `data/diccionario_datos.md`: la unidad de análisis es un **crédito desembolsado**, la evaluación ocurre antes de la decisión del comité y el corte de extracción es el **30/06/2025**. Los datos son sintéticos; estos hallazgos describen esta base y no representan a todas las MYPE peruanas.

Criterio general: antes del split solo se aplican reglas que **no estiman parámetros**: convertir tipos, unificar categorías mediante equivalencias explícitas, corregir unidades, eliminar duplicados exactos, marcar valores inválidos y excluir variables. La imputación, el escalado y la codificación se ajustarán en el pipeline del notebook 03, **solo con train**. Los cuantiles y cruces con la clase que aparecen aquí son diagnósticos; no se utilizan para elegir umbrales de limpieza ni para imputar por clase.

Las secciones 4.1–4.8 trabajan con copias de diagnóstico; la limpieza definitiva se aplica en 4.10. En 4.1 se cuentan las filas originales; desde 4.2 se reportan también los créditos únicos, y desde 4.3 todos los conteos corresponden a esa base sin duplicados. Se conservan las celdas y el orden de la plantilla. Por el alcance de esta entrega, las funciones se definen **dentro de este notebook**; su traslado a `src/` queda para el TF1.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RAIZ = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/raw/creditos_mype.csv").is_file()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Ejecutar desde la raíz del proyecto o desde notebooks/.")

TARGET = "mora_30d_6m"
FECHAS = ["fecha_solicitud", "fecha_desembolso"]
IDS = ["id_credito", "id_cliente"]
ENTEROS = [
    "edad", "num_dependientes", "antiguedad_negocio_meses",
    "num_creditos_previos_entidad", "max_dias_atraso_12m",
    "num_entidades_sistema", "num_consultas_buro_6m", "plazo_meses",
    "num_gestiones_cobranza", TARGET,
]
REALES = [
    "ingreso_mensual_declarado", "gasto_mensual_declarado", "score_buro",
    "deuda_total_sistema", "monto_solicitado", "monto_aprobado",
    "tasa_interes_anual", "cuota_mensual",
]
df_raw = pd.read_csv(RAIZ / "data/raw/creditos_mype.csv")
df = df_raw.copy(deep=True)
CATEGORICAS = [c for c in df_raw if c not in IDS + FECHAS + ENTEROS + REALES]
print(f"Entrada: {len(df_raw):,} filas y {df_raw.shape[1]} columnas")
display(df_raw.head(3))

Entrada: 15,579 filas y 35 columnas


,id_credito,id_cliente,fecha_solicitud,fecha_desembolso,region,codigo_asesor,canal_captacion,edad,sexo,estado_civil,nivel_educativo,tipo_vivienda,num_dependientes,rubro,antiguedad_negocio_meses,ingreso_mensual_declarado,gasto_mensual_declarado,cliente_recurrente,num_creditos_previos_entidad,score_buro,calificacion_sbs,max_dias_atraso_12m,num_entidades_sistema,deuda_total_sistema,num_consultas_buro_6m,monto_solicitado,monto_aprobado,plazo_meses,tasa_interes_anual,cuota_mensual,tiene_garantia,telefono_verificado,num_gestiones_cobranza,estado_credito_corte,mora_30d_6m
0,CR2012615,CL109437,2024-10-21,2024-10-29,Lima,AS039,Asesor de campo,59,Masculino,Casado,Primaria,Propia,0,Servicios,213.0000,3940.0,"1,700.0000",No,0,752.0000,Normal,9,4,"27,110.0000",5,"4,000.0000","4,000.0000",18,52.2900,304.9600,No,Si,0,Vigente,0
1,CR2002778,CL102078,2022-08-30,2022-09-04,Loreto,AS004,Asesor de campo,42,M,Casado,Tecnica,Alquilada,1,Servicios,44.0000,1830.0,"1,020.0000",Si,1,596.0000,Normal,45,2,"1,970.0000",0,"1,300.0000","1,100.0000",12,61.5300,117.7300,No,Si,0,Cancelado,0
2,CR2013067,CL109770,25/05/2024,09/06/2024,Loreto,AS015,Agencia,39,Masculino,Casado,Primaria,Alquilada,2,Agropecuario,71.0000,2410.0,"1,700.0000",No,0,NaN,Sin historial,0,0,0.0000,1,"1,900.0000","1,900.0000",12,57.1800,200.5800,No,Si,0,Cancelado,0


### 4.1 Tipos de datos

In [2]:
def convertir_fecha(serie):
    texto = serie.astype("string").str.strip()
    iso = pd.to_datetime(texto.where(texto.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)),
                         format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(texto.where(texto.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)),
                         format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)


def convertir_tipos(datos):
    salida = datos.copy(deep=True)
    for col in FECHAS:
        salida[col] = convertir_fecha(salida[col])
    texto = salida["ingreso_mensual_declarado"].astype("string").str.strip()
    moneda = texto.str.fullmatch(r"S/\s*\d{1,3}(?:,\d{3})*\.\d{2}", na=False)
    texto.loc[moneda] = (texto.loc[moneda].str.replace("S/", "", regex=False)
                        .str.replace(",", "", regex=False).str.strip())
    salida["ingreso_mensual_declarado"] = texto
    for col in ENTEROS + REALES:
        valores = pd.to_numeric(salida[col], errors="coerce")
        if col in ENTEROS:
            if (valores.notna() & valores.mod(1).ne(0)).any():
                raise ValueError(f"{col}: hay valores fraccionarios; revisar sin redondear.")
            salida[col] = valores.astype("Int64")
        else:
            salida[col] = valores.astype("float64")
    for col in IDS + CATEGORICAS:
        salida[col] = salida[col].astype("string")
    return salida


df_tipos = convertir_tipos(df_raw)
tabla_tipos = pd.DataFrame({
    "tipo_original": df_raw.dtypes.astype(str),
    "tipo_propuesto": df_tipos.dtypes.astype(str),
    "faltantes_originales": df_raw.isna().sum(),
    "nuevos_nulos_conversion": (df_raw.notna() & df_tipos.isna()).sum(),
})
display(tabla_tipos)
display(pd.DataFrame({
    col: {
        "formato_ISO": df_raw[col].str.fullmatch(r"\d{4}-\d{2}-\d{2}").sum(),
        "formato_dia_mes_anio": df_raw[col].str.fullmatch(r"\d{2}/\d{2}/\d{4}").sum(),
        "no_interpretables": df_tipos[col].isna().sum(),
    } for col in FECHAS
}).T)
ingreso_no_numerico = (df_raw["ingreso_mensual_declarado"].notna()
                       & pd.to_numeric(df_raw["ingreso_mensual_declarado"], errors="coerce").isna())
print("Ingresos con formato monetario:", int(ingreso_no_numerico.sum()))
display(df_raw.loc[ingreso_no_numerico, ["id_credito", "ingreso_mensual_declarado"]].head(6))
assert tabla_tipos["nuevos_nulos_conversion"].sum() == 0

,tipo_original,tipo_propuesto,faltantes_originales,nuevos_nulos_conversion
id_credito,str,string,0,0
id_cliente,str,string,0,0
fecha_solicitud,str,datetime64[us],0,0
fecha_desembolso,str,datetime64[us],0,0
region,str,string,0,0
codigo_asesor,str,string,0,0
canal_captacion,str,string,0,0
edad,int64,Int64,0,0
sexo,str,string,0,0
estado_civil,str,string,0,0


,formato_ISO,formato_dia_mes_anio,no_interpretables
fecha_solicitud,13689,1890,0
fecha_desembolso,13689,1890,0


Ingresos con formato monetario: 758


,id_credito,ingreso_mensual_declarado
23,CR2014750,"S/ 1,470.00"
24,CR2013701,"S/ 1,760.00"
57,CR2000625,"S/ 4,530.00"
72,CR2010009,"S/ 3,030.00"
89,CR2011760,"S/ 1,800.00"
104,CR2006180,"S/ 4,620.00"


**Evidencia:** La entrada contiene 15 579 filas y 35 columnas. Cada columna de fecha mezcla 13 689 valores `AAAA-MM-DD` y 1 890 `DD/MM/AAAA`. Hay 758 ingresos con texto monetario, por ejemplo `S/ 3,590.00`; una conversión numérica directa los perdería. La antigüedad aparece como decimal por sus faltantes, aunque todos sus valores observados son enteros. Ninguna conversión propuesta genera faltantes nuevos.

**Decisión:** Interpretar explícitamente ambos formatos de fecha; quitar el prefijo y los separadores de miles solo cuando el ingreso cumple el patrón monetario observado. Convertir medidas a números, conteos a enteros anulables `Int64` e identificadores/categorías a texto. No redondear conteos fraccionarios de manera silenciosa.

**Justificación:** Las operaciones temporales y aritméticas requieren tipos adecuados. Especificar los formatos evita intercambiar día y mes; `Int64` permite representar una cantidad entera y un faltante sin usar códigos como −1. La conversión de tipos no convierte automáticamente un valor en válido: los dominios se revisan en 4.5.

### 4.2 Duplicados

In [3]:
sin_duplicados = df_raw.drop_duplicates(keep="first")
tabla_duplicados = pd.Series({
    "filas_originales": len(df_raw),
    "copias_exactas_adicionales": int(df_raw.duplicated().sum()),
    "filas_tras_quitar_copias": len(sin_duplicados),
    "ids_credito_repetidos_antes": int(df_raw["id_credito"].duplicated().sum()),
    "ids_credito_repetidos_despues": int(sin_duplicados["id_credito"].duplicated().sum()),
    "clientes_distintos": sin_duplicados["id_cliente"].nunique(),
    "clientes_con_varios_creditos": int(sin_duplicados.groupby("id_cliente").size().gt(1).sum()),
    "creditos_adicionales_del_mismo_cliente": int(sin_duplicados["id_cliente"].duplicated().sum()),
}, name="cantidad")
display(tabla_duplicados.to_frame())
display(df_raw.loc[df_raw.duplicated(keep=False)].sort_values("id_credito").head(6))
conflictos_id = sin_duplicados.loc[sin_duplicados["id_credito"].duplicated(keep=False)]
assert conflictos_id.empty, "Un mismo crédito tiene versiones diferentes; requiere revisión."
df_base = convertir_tipos(sin_duplicados)

,cantidad
filas_originales,15579
copias_exactas_adicionales,184
filas_tras_quitar_copias,15395
ids_credito_repetidos_antes,184
ids_credito_repetidos_despues,0
clientes_distintos,11500
clientes_con_varios_creditos,3297
creditos_adicionales_del_mismo_cliente,3895


,id_credito,id_cliente,fecha_solicitud,fecha_desembolso,region,codigo_asesor,canal_captacion,edad,sexo,estado_civil,nivel_educativo,tipo_vivienda,num_dependientes,rubro,antiguedad_negocio_meses,ingreso_mensual_declarado,gasto_mensual_declarado,cliente_recurrente,num_creditos_previos_entidad,score_buro,calificacion_sbs,max_dias_atraso_12m,num_entidades_sistema,deuda_total_sistema,num_consultas_buro_6m,monto_solicitado,monto_aprobado,plazo_meses,tasa_interes_anual,cuota_mensual,tiene_garantia,telefono_verificado,num_gestiones_cobranza,estado_credito_corte,mora_30d_6m
12839,CR2000117,CL100092,2022-08-02,2022-08-10,Loreto,AS015,Agencia,38,M,Soltero,Secundaria,Alquilada,2,Transporte,39.0000,2880.0,"2,380.0000",No,0,675.0000,Normal,16,2,"2,720.0000",1,"18,300.0000","12,300.0000",18,59.6500,969.6800,No,NaN,0,Cancelado,0
2303,CR2000117,CL100092,2022-08-02,2022-08-10,Loreto,AS015,Agencia,38,M,Soltero,Secundaria,Alquilada,2,Transporte,39.0000,2880.0,"2,380.0000",No,0,675.0000,Normal,16,2,"2,720.0000",1,"18,300.0000","12,300.0000",18,59.6500,969.6800,No,NaN,0,Cancelado,0
6694,CR2000165,CL100127,2022-04-04,2022-04-08,Lambayeque,AS051,Asesor de campo,45,F,Viudo,Tecnica,Alquilada,1,Servicios,60.0000,1000.0,600.0000,No,0,511.0000,Normal,0,0,0.0000,3,800.0000,800.0000,6,65.4200,154.0200,No,Si,0,Cancelado,0
8916,CR2000165,CL100127,2022-04-04,2022-04-08,Lambayeque,AS051,Asesor de campo,45,F,Viudo,Tecnica,Alquilada,1,Servicios,60.0000,1000.0,600.0000,No,0,511.0000,Normal,0,0,0.0000,3,800.0000,800.0000,6,65.4200,154.0200,No,Si,0,Cancelado,0
10184,CR2000263,CL100204,2022-01-27,2022-02-09,Junin,AS055,Asesor de campo,46,F,Casado,Primaria,Propia,2,Comercio,44.0000,2960.0,"1,830.0000",No,0,727.0000,Normal,3,0,0.0000,2,"1,500.0000","1,500.0000",3,0.5252,536.2200,No,Si,1,Cancelado,0
7679,CR2000263,CL100204,2022-01-27,2022-02-09,Junin,AS055,Asesor de campo,46,F,Casado,Primaria,Propia,2,Comercio,44.0000,2960.0,"1,830.0000",No,0,727.0000,Normal,3,0,0.0000,2,"1,500.0000","1,500.0000",3,0.5252,536.2200,No,Si,1,Cancelado,0


**Evidencia:** Hay 184 copias exactas adicionales. Al retirarlas quedan 15 395 créditos con `id_credito` único: no existen versiones distintas del mismo crédito. Los 11 500 clientes incluyen 3 297 con varios créditos; hay 3 895 créditos adicionales de clientes ya presentes.

**Decisión:** Eliminar solo duplicados de fila completa, conservando la primera copia idéntica, antes de separar los conjuntos. No deduplicar por cliente. Detener la limpieza si aparecieran versiones contradictorias de un mismo `id_credito`.

**Justificación:** Las copias dan peso artificial a ciertos créditos y podrían quedar a ambos lados del split. Varios préstamos del mismo cliente son observaciones legítimas. Mantener `id_cliente` como metadato permitirá revisar el solapamiento de clientes en el notebook 03; no se usará como predictor.

### 4.3 Categorías inconsistentes

In [4]:
def clave_categoria(valor):
    if pd.isna(valor):
        return pd.NA
    texto = unicodedata.normalize("NFKD", str(valor).strip().casefold())
    return " ".join("".join(c for c in texto if not unicodedata.combining(c)).split())


MAPAS = {
    "region": {
        "lima": "Lima", "lima metropolitana": "Lima", "piura": "Piura",
        "arequipa": "Arequipa", "la libertad": "La Libertad", "junin": "Junin",
        "cusco": "Cusco", "cuzco": "Cusco", "lambayeque": "Lambayeque",
        "puno": "Puno", "loreto": "Loreto",
    },
    "rubro": {
        "comercio": "Comercio", "comerc.": "Comercio", "servicio": "Servicios",
        "servicios": "Servicios", "agro": "Agropecuario", "agropecuario": "Agropecuario",
        "produccion": "Produccion", "transporte": "Transporte",
    },
    "sexo": {"f": "Femenino", "femenino": "Femenino", "m": "Masculino", "masculino": "Masculino"},
}


def unificar_categorias(datos):
    salida = datos.copy(deep=True)
    for col in CATEGORICAS:
        salida[col] = salida[col].str.strip().replace("", pd.NA)
    for col, mapa in MAPAS.items():
        claves = salida[col].map(clave_categoria)
        desconocidas = salida.loc[salida[col].notna() & ~claves.isin(mapa), col].unique()
        if len(desconocidas):
            raise ValueError(f"Categorías no documentadas en {col}: {desconocidas}")
        salida[col] = claves.map(mapa).astype("string")
    return salida


df_diag = unificar_categorias(df_base)
display(pd.DataFrame({
    "categorias_antes": df_base[CATEGORICAS].nunique(),
    "categorias_unificadas": df_diag[CATEGORICAS].nunique(),
    "valores_originales": [sorted(df_base[c].dropna().unique().tolist()) for c in CATEGORICAS],
}))
for col in MAPAS:
    display(pd.DataFrame({"original": df_base[col], "unificada": df_diag[col]})
            .value_counts().rename("creditos").reset_index())
display(pd.crosstab(df_base["fecha_desembolso"].dt.to_period("Q"), df_base["sexo"]))

,categorias_antes,categorias_unificadas,valores_originales
region,25,9,"[AREQUIPA, Arequipa, Arequipa , CUSCO, Cusco, ..."
codigo_asesor,45,45,"[AS001, AS002, AS003, AS004, AS006, AS007, AS0..."
canal_captacion,3,3,"[Agencia, Asesor de campo, Digital]"
sexo,4,2,"[F, Femenino, M, Masculino]"
estado_civil,5,5,"[Casado, Conviviente, Divorciado, Soltero, Viudo]"
nivel_educativo,4,4,"[Primaria, Secundaria, Tecnica, Universitaria]"
tipo_vivienda,3,3,"[Alquilada, Familiar, Propia]"
rubro,16,5,"[AGROPECUARIO, Agro, Agropecuario, COMERCIO, C..."
cliente_recurrente,2,2,"[No, Si]"
calificacion_sbs,6,6,"[CPP, Deficiente, Dudoso, Normal, Perdida, Sin..."


,original,unificada,creditos
0,Lima,Lima,3907
1,Piura,Piura,1550
2,La Libertad,La Libertad,1335
3,Arequipa,Arequipa,1329
4,Junin,Junin,1222
5,Cusco,Cusco,1058
6,Lambayeque,Lambayeque,1036
7,Puno,Puno,946
8,Loreto,Loreto,857
9,LIMA,Lima,213


,original,unificada,creditos
0,Comercio,Comercio,6095
1,Servicios,Servicios,2786
2,Agropecuario,Agropecuario,1680
3,Transporte,Transporte,1648
4,Produccion,Produccion,1647
5,COMERCIO,Comercio,235
6,comercio,Comercio,233
7,Comerc.,Comercio,201
8,Servicio,Servicios,162
9,SERVICIOS,Servicios,154


,original,unificada,creditos
0,Femenino,Femenino,4838
1,Masculino,Masculino,3928
2,F,Femenino,3700
3,M,Masculino,2929


sexo,F,Femenino,M,Masculino
fecha_desembolso,,,,
2022Q1,423,0,329,0
2022Q2,577,0,423,0
2022Q3,626,0,500,0
2022Q4,671,0,539,0
2023Q1,688,0,547,0
2023Q2,715,0,591,0
2023Q3,0,769,0,598
2023Q4,0,787,0,618
2024Q1,0,764,0,671


**Evidencia:** `region` tiene 25 etiquetas para 9 regiones; `rubro`, 16 para 5 actividades; `sexo`, 4 para 2 categorías. Se observan espacios, mayúsculas, tildes y alias (`Cuzco/Cusco`, `Comerc./Comercio`, `Agro/Agropecuario`). Hasta 2023Q2 el sexo se registra como `F/M`; desde 2023Q3, como `Femenino/Masculino`. El resto de las categóricas no muestra variantes equivalentes adicionales.

**Decisión:** Usar los mapas explícitos mostrados arriba, con normalización auxiliar de espacios, tildes y mayúsculas. Interpretar `Lima Metropolitana` como `Lima` porque la variable describe una región; dejar esta equivalencia como supuesto revisable. Conservar `DIGITAL` y `Sin historial`: tienen significado propio. Mantener los faltantes.

**Justificación:** Etiquetas equivalentes fragmentan los grupos y generan columnas redundantes. El cambio de codificación del sexo podría actuar como un indicador accidental del período e introducir categorías desconocidas en una evaluación temporal. El mapeo usa significados, no tasas de mora ni frecuencias aprendidas. Una etiqueta futura no documentada exige revisar el mapa.

### 4.4 Valores faltantes

In [5]:
tabla_faltantes = pd.DataFrame({
    "cantidad": df_diag.isna().sum(),
    "porcentaje": df_diag.isna().mean().mul(100),
}).sort_values("cantidad", ascending=False)
display(tabla_faltantes)
cols_faltantes = tabla_faltantes.index[tabla_faltantes["cantidad"].gt(0)].tolist()
for nombre, grupos in {
    "canal": df_diag["canal_captacion"],
    "region_unificada": df_diag["region"],
    "anio_desembolso": df_diag["fecha_desembolso"].dt.year,
    "clase_solo_diagnostico": df_diag[TARGET],
}.items():
    print(f"Porcentaje de faltantes por {nombre}")
    tasas = df_diag[cols_faltantes].isna().groupby(grupos, dropna=False).mean().mul(100)
    tasas.insert(0, "n_creditos", df_diag.groupby(grupos, dropna=False).size())
    display(tasas.round(2))
display(pd.crosstab(df_diag["calificacion_sbs"], df_diag["score_buro"].isna(),
                    colnames=["score_ausente"], margins=True))
print("Créditos con al menos un faltante:", int(df_diag.isna().any(axis=1).sum()))
assert df_diag["score_buro"].isna().equals(df_diag["calificacion_sbs"].eq("Sin historial").astype(bool))

,cantidad,porcentaje
score_buro,2377,15.4401
nivel_educativo,1265,8.2170
antiguedad_negocio_meses,663,4.3066
gasto_mensual_declarado,615,3.9948
telefono_verificado,461,2.9945
ingreso_mensual_declarado,384,2.4943
id_credito,0,0.0000
fecha_solicitud,0,0.0000
id_cliente,0,0.0000
canal_captacion,0,0.0000


Porcentaje de faltantes por canal


,n_creditos,score_buro,nivel_educativo,antiguedad_negocio_meses,gasto_mensual_declarado,telefono_verificado,ingreso_mensual_declarado
canal_captacion,,,,,,,
Agencia,5344,15.1900,1.9500,0.0000,4.0000,3.1100,2.3200
Asesor de campo,8148,15.8800,13.7300,0.0000,4.0500,2.9500,2.7600
Digital,1903,14.2400,2.2100,34.8400,3.7300,2.8900,1.8400


Porcentaje de faltantes por region_unificada


,n_creditos,score_buro,nivel_educativo,antiguedad_negocio_meses,gasto_mensual_declarado,telefono_verificado,ingreso_mensual_declarado
region,,,,,,,
Arequipa,1549,14.5900,7.9400,4.0700,4.5200,2.6500,2.0000
Cusco,1216,15.8700,8.9600,3.7000,3.2100,3.7000,2.6300
Junin,1428,16.8800,9.3800,4.2700,3.7800,3.0100,2.0300
La Libertad,1546,13.9700,7.3700,5.2400,4.4000,3.4900,2.9100
Lambayeque,1208,15.7300,8.5300,3.8100,5.1300,3.3100,2.6500
Lima,4540,16.1000,8.2200,4.3000,3.9600,2.9500,2.3600
Loreto,962,15.2800,7.2800,4.3700,2.8100,2.9100,2.9100
Piura,1840,14.6200,8.1500,4.4600,3.9700,2.7700,2.6600
Puno,1106,14.8300,8.0500,4.3400,3.8000,2.2600,2.8000


...

,n_creditos,score_buro,nivel_educativo,antiguedad_negocio_meses,gasto_mensual_declarado,telefono_verificado,ingreso_mensual_declarado
fecha_desembolso,,,,,,,
2022,4088,19.3700,8.9300,1.8600,3.5200,2.8600,2.5000
2023,5313,15.9800,8.4900,4.2200,4.1000,3.1800,2.5000
2024,5994,12.2800,7.4900,6.0600,4.2200,2.9200,2.4900


Porcentaje de faltantes por clase_solo_diagnostico


,n_creditos,score_buro,nivel_educativo,antiguedad_negocio_meses,gasto_mensual_declarado,telefono_verificado,ingreso_mensual_declarado
mora_30d_6m,,,,,,,
0,13716,15.9100,8.3700,4.1100,3.9400,3.0000,2.4700
1,1679,11.6100,6.9700,5.9000,4.4100,2.9200,2.6800


score_ausente,False,True,All
calificacion_sbs,,,
CPP,1747,0,1747
Deficiente,796,0,796
Dudoso,412,0,412
Normal,9817,0,9817
Perdida,246,0,246
Sin historial,0,2377,2377
All,13018,2377,15395


Créditos con al menos un faltante: 5060


**Evidencia:** En los créditos únicos faltan: `score_buro` en 2 377 (15,44 %), `nivel_educativo` en 1 265 (8,22 %), antigüedad en 663 (4,31 %), gastos en 615 (3,99 %), teléfono verificado en 461 (2,99 %) e ingresos en 384 (2,49 %). Los 2 377 scores ausentes coinciden exactamente con `Sin historial`. Toda la antigüedad ausente está en Digital: falta en aproximadamente 34,8 % de ese canal. La educación falta en 13,7 % del canal de campo, frente a cerca de 2 % en los otros canales. Hay 5 060 créditos con algún faltante. Las tablas incluyen también región, año y clase; las diferencias descriptivas no prueban un mecanismo causal de ausencia.

**Decisión:** Conservar las filas y sus nulos. Agregar indicadores de ausencia de score, antigüedad, ingresos, gastos, educación y verificación telefónica después de marcar errores de dominio. Preservar `Sin historial`. Dejar para el notebook 03 la imputación numérica con mediana aprendida en train y la categórica con una categoría explícita `No informado`, seguida de la codificación. Los indicadores conservan la distinción aun si se necesita un número de reemplazo para el algoritmo.

**Justificación:** El score presenta un **faltante estructural**: no equivale a un score de cero ni al promedio de un cliente con historial. La ausencia de antigüedad parece asociada al proceso digital, pero el diccionario no confirma que el campo no aplique: se considera faltante problemático de captura. Borrar filas completas eliminaría casi un tercio de los créditos y alteraría la mezcla de canales y perfiles. Un teléfono sin dato tampoco implica `No`. No se imputa aquí ni se utiliza la clase para completar valores.

### 4.5 Valores imposibles y extremos

In [6]:
def mascaras_invalidos(datos):
    reglas = {
        "edad": ~datos["edad"].between(18, 120),
        "antiguedad_negocio_meses": datos["antiguedad_negocio_meses"].lt(0),
        "plazo_meses": datos["plazo_meses"].le(0),
        "score_buro": ~datos["score_buro"].between(300, 950),
        "tasa_interes_anual": datos["tasa_interes_anual"].le(0),
    }
    for col in ["ingreso_mensual_declarado", "monto_solicitado", "monto_aprobado", "cuota_mensual"]:
        reglas[col] = datos[col].le(0)
    for col in ["gasto_mensual_declarado", "deuda_total_sistema", "num_dependientes",
                "num_creditos_previos_entidad", "max_dias_atraso_12m", "num_entidades_sistema",
                "num_consultas_buro_6m", "num_gestiones_cobranza"]:
        reglas[col] = datos[col].lt(0)
    return {col: (mascara & datos[col].notna()).fillna(False) for col, mascara in reglas.items()}


invalidos = mascaras_invalidos(df_diag)
display(pd.DataFrame({
    "minimo": df_diag[ENTEROS + REALES].min(),
    "mediana": df_diag[ENTEROS + REALES].median(),
    "maximo": df_diag[ENTEROS + REALES].max(),
    "invalidos_segun_regla": pd.Series({c: int(m.sum()) for c, m in invalidos.items()}),
}))
display(df_diag.loc[df_diag["edad"].isin([15, 999]), "edad"].value_counts().to_frame("creditos"))
MONETARIAS = ["ingreso_mensual_declarado", "gasto_mensual_declarado", "deuda_total_sistema",
              "monto_solicitado", "monto_aprobado", "cuota_mensual"]
extremos = []
for col in MONETARIAS:
    q1, q3 = df_diag[col].quantile([0.25, 0.75])
    inferior, superior = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    extremos.append({"variable": col, "limite_IQR_inferior": inferior,
                     "limite_IQR_superior": superior,
                     "fuera_del_IQR": int((df_diag[col].lt(inferior) | df_diag[col].gt(superior)).sum()),
                     "p99": df_diag[col].quantile(0.99), "maximo": df_diag[col].max()})
display(pd.DataFrame(extremos).set_index("variable"))
display(df_diag.nlargest(5, "ingreso_mensual_declarado")
        [["id_credito", "ingreso_mensual_declarado", "gasto_mensual_declarado", "monto_solicitado"]])

,minimo,mediana,maximo,invalidos_segun_regla
antiguedad_negocio_meses,-1.0000,50.0000,365.0000,46.0000
cuota_mensual,31.6000,613.1500,"10,224.2700",0.0000
deuda_total_sistema,0.0000,"3,100.0000","155,990.0000",0.0000
edad,15.0000,41.0000,999.0000,76.0000
gasto_mensual_declarado,60.0000,"1,830.0000","19,880.0000",0.0000
ingreso_mensual_declarado,390.0000,"3,400.0000","143,100.0000",0.0000
max_dias_atraso_12m,0.0000,0.0000,180.0000,0.0000
monto_aprobado,400.0000,"6,100.0000","67,300.0000",0.0000
monto_solicitado,500.0000,"6,500.0000","60,000.0000",0.0000
mora_30d_6m,0.0000,0.0000,1.0000,NaN


,creditos
edad,
999,46
15,30


,limite_IQR_inferior,limite_IQR_superior,fuera_del_IQR,p99,maximo
variable,,,,,
ingreso_mensual_declarado,"-2,015.0000","9,385.0000",720,"15,059.0000","143,100.0000"
gasto_mensual_declarado,"-1,415.0000","5,425.0000",661,"8,150.5000","19,880.0000"
deuda_total_sistema,"-12,150.0000","20,250.0000",1076,"46,784.8000","155,990.0000"
monto_solicitado,"-7,700.0000","22,700.0000",909,"40,900.0000","60,000.0000"
monto_aprobado,"-7,000.0000","21,000.0000",979,"38,906.0000","67,300.0000"
cuota_mensual,-486.5925,"1,829.4275",990,"3,279.2796","10,224.2700"


,id_credito,ingreso_mensual_declarado,gasto_mensual_declarado,monto_solicitado
10933,CR2000817,"143,100.0000","11,790.0000","5,100.0000"
8396,CR2007760,"129,700.0000","4,430.0000","10,500.0000"
9654,CR2013766,"100,800.0000","6,000.0000","8,200.0000"
15156,CR2010147,"97,600.0000","6,030.0000","19,600.0000"
6595,CR2004508,"95,700.0000","3,010.0000","26,900.0000"


**Evidencia:** Se registran 46 edades de 999 años, 30 de 15 años, 46 antigüedades de −1 mes y 30 plazos de cero meses. El score observado está entre 300 y 950, dentro del diccionario. No hay montos o ingresos no positivos, ni gastos/deudas negativos. El ingreso máximo es S/ 143 100 y la deuda máxima S/ 155 990. La regla exploratoria de 1,5 IQR señalaría 720 ingresos y 1 076 deudas, pero por sí sola no demuestra errores.

**Decisión:** Marcar como faltantes las edades fuera de [18, 120], las antigüedades negativas y los plazos no positivos; las otras reglas de dominio quedan expresadas y verificadas. Para edad, 999 es un error evidente; tratar 15 como inválido supone que el titular del producto es adulto, algo que debe confirmarse con el área. Conservar los extremos monetarios positivos sin recorte ni eliminación de filas. La tasa se revisa primero como posible problema de unidad en 4.6.

**Justificación:** No se conoce el valor real que reemplazaría a 999, −1 o 0; asignar una edad o plazo típico inventaría información. Un plazo cero no permite interpretar cuotas mensuales. Los ingresos y deudas pueden tener colas largas; recortarlos con cuantiles de toda la base eliminaría posibles casos reales y estimaría una transformación antes del split. El rango de edad es una regla de revisión fija, no una estimación de esta muestra ni una política de crédito validada. No se fija una edad máxima de 65 o 75 sin sustento.

### 4.6 Unidades inconsistentes

In [7]:
tasa_fraccion = df_diag["tasa_interes_anual"].between(0, 1, inclusive="neither")
tasa_porcentaje = df_diag["tasa_interes_anual"].where(~tasa_fraccion, df_diag["tasa_interes_anual"] * 100)
display(df_diag.assign(escala=np.where(tasa_fraccion, "posible_fraccion", "porcentaje"))
        .groupby("escala")["tasa_interes_anual"].agg(["count", "min", "median", "max"]))


def cuota_teorica(datos, tasa_en_porcentaje):
    mensual = (1 + tasa_en_porcentaje / 100) ** (1 / 12) - 1
    plazo = datos["plazo_meses"].where(datos["plazo_meses"].gt(0)).astype(float)
    return datos["monto_aprobado"] * mensual / (1 - (1 + mensual) ** (-plazo))


verificables = (tasa_fraccion & df_diag["plazo_meses"].gt(0)
               & df_diag["monto_aprobado"].le(df_diag["monto_solicitado"]))
contraste_tasa = []
for interpretacion, tasa in {"literal_como_porcentaje": df_diag["tasa_interes_anual"],
                            "fraccion_multiplicada_por_100": tasa_porcentaje}.items():
    error = ((cuota_teorica(df_diag, tasa) - df_diag["cuota_mensual"]).abs()
             / df_diag["cuota_mensual"]).loc[verificables]
    contraste_tasa.append({"interpretacion": interpretacion, "casos": len(error),
                          "error_relativo_mediano_pct": error.median() * 100,
                          "error_relativo_maximo_pct": error.max() * 100,
                          "cuotas_con_error_menor_1pct": int(error.lt(0.01).sum())})
display(pd.DataFrame(contraste_tasa).set_index("interpretacion"))
display(pd.DataFrame({"tasa_original": df_diag.loc[tasa_fraccion, "tasa_interes_anual"],
                      "tasa_en_porcentaje": tasa_porcentaje.loc[tasa_fraccion]}).head(6))

,count,min,median,max
escala,,,,
porcentaje,15242,22.0000,58.5250,85.6400
posible_fraccion,153,0.2851,0.5899,0.7826


,casos,error_relativo_mediano_pct,error_relativo_maximo_pct,cuotas_con_error_menor_1pct
interpretacion,,,,
literal_como_porcentaje,152,21.5090,48.2599,0
fraccion_multiplicada_por_100,152,0.0004,0.0052,152


,tasa_original,tasa_en_porcentaje
62,0.6811,68.1100
154,0.6368,63.6800
225,0.6295,62.9500
267,0.5445,54.4500
313,0.7826,78.2600
401,0.6077,60.7700


**Evidencia:** Hay 153 tasas entre 0,2851 y 0,7826, mientras las otras 15 242 van de 22 a 85,64. El diccionario expresa la tasa efectiva anual en porcentaje. Para contrastar la hipótesis se usa una anualidad mensual: tasa mensual = `(1 + TEA/100)^(1/12) − 1`. En 152 casos con plazo válido y sin contradicción de montos, interpretar literalmente la tasa produce un error mediano de cuota del 21,51 %; multiplicarla por 100 reduce ese error a aproximadamente 0,00042 %. Las 152 cuotas quedan a menos de 1 % de la fórmula. El caso restante tiene una contradicción de montos y no se usa para esta validación.

**Decisión:** Convertir a porcentaje las tasas que cumplen `0 < tasa < 1`, multiplicando por 100. No cambiar las demás. La regla y la evidencia se conservan en la auditoría aunque la tasa final se excluya como predictor en 4.8. No dividir ingresos extremos entre 12 ni asumir otra moneda: no hay evidencia de que sean ingresos anuales o dólares.

**Justificación:** La unidad declarada y la coherencia con la cuota respaldan una inconsistencia de escala; no se usa un promedio para corregirla. La regla es determinista y se limita al patrón detectado. Su generalización exigiría confirmar que el producto no ofrece tasas reales inferiores a 1 % anual. La fórmula es un contraste de coherencia, no un supuesto para recalcular contratos con seguros, comisiones o cronogramas especiales.

### 4.7 Reglas de negocio

In [8]:
def revisar_reglas(datos):
    edad_valida = datos["edad"].between(18, 120)
    reglas = {
        "desembolso_anterior_a_solicitud": datos["fecha_desembolso"].lt(datos["fecha_solicitud"]),
        "desembolso_fuera_2022_2024": ~datos["fecha_desembolso"].between("2022-01-01", "2024-12-31"),
        "sin_seis_meses_al_corte": (datos["fecha_desembolso"] + pd.DateOffset(months=6)).gt(pd.Timestamp("2025-06-30")),
        "objetivo_fuera_0_1_o_ausente": ~datos[TARGET].isin([0, 1]),
        "aprobado_mayor_solicitado": datos["monto_aprobado"].gt(datos["monto_solicitado"]),
        "negocio_mayor_edad_registrada": datos["antiguedad_negocio_meses"].gt(datos["edad"] * 12),
        "negocio_previo_mayoria_edad_con_edad_valida": edad_valida & datos["antiguedad_negocio_meses"].gt((datos["edad"] - 18) * 12),
        "recurrente_incompatible_con_creditos_previos": datos["cliente_recurrente"].eq("Si").ne(datos["num_creditos_previos_entidad"].gt(0)),
        "cero_entidades_con_deuda": datos["num_entidades_sistema"].eq(0) & datos["deuda_total_sistema"].gt(0),
        "entidades_con_deuda_cero": datos["num_entidades_sistema"].gt(0) & datos["deuda_total_sistema"].eq(0),
        "canal_digital_incompatible_con_asesor": datos["canal_captacion"].eq("Digital").ne(datos["codigo_asesor"].eq("DIGITAL")),
        "score_ausente_no_coincide_con_sin_historial": datos["score_buro"].isna().ne(datos["calificacion_sbs"].eq("Sin historial")),
        "gastos_superan_ingresos": datos["gasto_mensual_declarado"].gt(datos["ingreso_mensual_declarado"]),
        "cuota_supera_ingresos": datos["cuota_mensual"].gt(datos["ingreso_mensual_declarado"]),
    }
    return pd.DataFrame({nombre: mascara.fillna(False) for nombre, mascara in reglas.items()}, index=datos.index)


reglas_antes = revisar_reglas(df_diag)
display(pd.DataFrame({"casos": reglas_antes.sum(), "porcentaje": reglas_antes.mean() * 100}))
for regla, columnas in {
    "desembolso_anterior_a_solicitud": ["fecha_solicitud", "fecha_desembolso"],
    "aprobado_mayor_solicitado": ["monto_solicitado", "monto_aprobado", "cuota_mensual"],
    "negocio_mayor_edad_registrada": ["edad", "antiguedad_negocio_meses"],
    "cuota_supera_ingresos": ["ingreso_mensual_declarado", "cuota_mensual"],
}.items():
    print(regla)
    display(df_diag.loc[reglas_antes[regla], ["id_credito"] + columnas].head(5))

,casos,porcentaje
desembolso_anterior_a_solicitud,76,0.4937
desembolso_fuera_2022_2024,0,0.0000
sin_seis_meses_al_corte,0,0.0000
objetivo_fuera_0_1_o_ausente,0,0.0000
aprobado_mayor_solicitado,123,0.7990
negocio_mayor_edad_registrada,1,0.0065
negocio_previo_mayoria_edad_con_edad_valida,49,0.3183
recurrente_incompatible_con_creditos_previos,0,0.0000
cero_entidades_con_deuda,0,0.0000
entidades_con_deuda_cero,0,0.0000


desembolso_anterior_a_solicitud


,id_credito,fecha_solicitud,fecha_desembolso
60,CR2007118,2023-11-17,2023-11-14
79,CR2004496,2023-01-10,2023-01-02
382,CR2013332,2023-10-17,2023-10-11
814,CR2000988,2024-07-01,2024-06-30
906,CR2010437,2022-12-24,2022-12-20


aprobado_mayor_solicitado


,id_credito,monto_solicitado,monto_aprobado,cuota_mensual
110,CR2008500,"9,600.0000","13,600.0000",513.8700
495,CR2003000,"7,200.0000","9,500.0000",461.7700
644,CR2005349,"5,200.0000","6,900.0000",562.7600
709,CR2009542,"20,300.0000","23,300.0000",635.1500
714,CR2005769,"4,600.0000","6,200.0000",491.1000


negocio_mayor_edad_registrada


,id_credito,edad,antiguedad_negocio_meses
8949,CR2009771,15,224


cuota_supera_ingresos


,id_credito,ingreso_mensual_declarado,cuota_mensual
168,CR2010455,"1,710.0000","2,428.4500"
280,CR2001680,"2,380.0000","2,509.0100"
308,CR2012673,"1,250.0000","1,339.2000"
439,CR2007341,"3,970.0000","4,248.2300"
528,CR2002938,"4,120.0000","5,375.3800"


**Evidencia:** Hay 76 desembolsos anteriores a la solicitud y 123 montos aprobados mayores que los solicitados. En un crédito la antigüedad supera la edad registrada, pero esa edad es 15 y ya se ha marcado como dudosa. En 49 titulares con edad válida el negocio habría comenzado antes de sus 18 años. En 46 créditos la cuota supera el ingreso declarado; no hay gastos mayores al ingreso entre pares observados. No se detectan contradicciones entre recurrencia y créditos previos, deuda y número de entidades, canal digital y asesor, ni score ausente y falta de historial. Todos los desembolsos caen en 2022–2024, tienen seis meses de observación al corte y el objetivo está completo y es binario.

**Decisión:** Separar las restricciones definitorias de las alertas que admiten contexto. Mantener las 76 parejas de fechas y las 123 parejas de montos originales, identificadas en las tablas de auditoría: no intercambiar fechas ni igualar montos sin saber qué campo está mal. Las condiciones finales se excluirán del conjunto de predictores por la decisión temporal de 4.8. No alterar antigüedades positivas por la edad de inicio ni corregir cuotas altas. En las reglas entre dos campos, un faltante significa que la comparación no puede verificarse; no significa que la regla se haya cumplido.

**Justificación:** Una solicitud debe preceder al desembolso según el proceso descrito, pero el error podría estar en cualquiera de las dos fechas. Los montos podrían reflejar una renegociación no documentada. Un negocio puede ser familiar y anterior a la participación del titular. Cuota mayor al ingreso es una alerta de capacidad de pago, no un error aritmético demostrado. No se impone una relación exacta entre la peor calificación de los últimos 12 meses y el atraso máximo sin conocer su regla de cálculo. Se conservan los casos para evitar borrar precisamente perfiles de riesgo. El split temporal por desembolso requerirá revisar la sensibilidad a las 76 contradicciones de fechas.

### 4.8 Variables con fuga de información

In [9]:
FUGA_CONFIRMADA = ["num_gestiones_cobranza", "estado_credito_corte"]
CONDICIONES_FINALES = ["monto_aprobado", "plazo_meses", "tasa_interes_anual", "cuota_mensual"]
EXCLUIR_SALIDA = FUGA_CONFIRMADA + CONDICIONES_FINALES

display(pd.crosstab(df_diag["estado_credito_corte"], df_diag[TARGET], margins=True))
display(pd.crosstab(df_diag["estado_credito_corte"], df_diag[TARGET], normalize="index").mul(100))
display(pd.crosstab(df_diag["num_gestiones_cobranza"], df_diag[TARGET], normalize="index").mul(100))
display(df_diag.groupby(TARGET)["num_gestiones_cobranza"].agg(["count", "mean", "median", "max"]))
display(pd.DataFrame([
    {"variables": ", ".join(FUGA_CONFIRMADA), "disponibilidad": "Posterior al crédito",
     "uso": "Excluir: fuga temporal confirmada"},
    {"variables": ", ".join(CONDICIONES_FINALES), "disponibilidad": "Condiciones pactadas; propuesta previa no acreditada",
     "uso": "Excluir por precaución para el objetivo predecisión"},
    {"variables": ", ".join(IDS + FECHAS), "disponibilidad": "Identificadores y fechas del proceso",
     "uso": "Solo metadatos de auditoría/split, fuera de X"},
    {"variables": TARGET, "disponibilidad": "Resultado de los primeros seis meses",
     "uso": "Solo y, nunca predictor"},
    {"variables": "score_buro, calificacion_sbs, max_dias_atraso_12m, deuda_total_sistema, num_consultas_buro_6m",
     "disponibilidad": "Historial previo, según el diccionario",
     "uso": "Conservar; confirmar que la extracción sea a fecha de evaluación"},
]))

mora_30d_6m,0,1,All
estado_credito_corte,,,
Cancelado,9539,337,9876
Castigado,0,210,210
Judicial,0,207,207
Refinanciado,0,436,436
Vencido,0,489,489
Vigente,4177,0,4177
All,13716,1679,15395


mora_30d_6m,0,1
estado_credito_corte,,
Cancelado,96.5877,3.4123
Castigado,0.0000,100.0000
Judicial,0.0000,100.0000
Refinanciado,0.0000,100.0000
Vencido,0.0000,100.0000
Vigente,100.0000,0.0000


mora_30d_6m,0,1
num_gestiones_cobranza,,
0,99.9782,0.0218
1,99.5736,0.4264
2,94.4669,5.5331
3,56.3830,43.6170
4,4.4944,95.5056
5,0.0000,100.0000
6,0.0000,100.0000
7,0.0000,100.0000
8,0.0000,100.0000


,count,mean,median,max
mora_30d_6m,,,,
0,13716,0.4000,0.0000,4
1,1679,6.9863,7.0000,18


,variables,disponibilidad,uso
0,"num_gestiones_cobranza, estado_credito_corte",Posterior al crédito,Excluir: fuga temporal confirmada
1,"monto_aprobado, plazo_meses, tasa_interes_anua...",Condiciones pactadas; propuesta previa no acre...,Excluir por precaución para el objetivo predec...
2,"id_credito, id_cliente, fecha_solicitud, fecha...",Identificadores y fechas del proceso,"Solo metadatos de auditoría/split, fuera de X"
3,mora_30d_6m,Resultado de los primeros seis meses,"Solo y, nunca predictor"
4,"score_buro, calificacion_sbs, max_dias_atraso_...","Historial previo, según el diccionario",Conservar; confirmar que la extracción sea a f...


**Evidencia:** `num_gestiones_cobranza` y `estado_credito_corte` se acumulan hasta el 30/06/2025: no existen cuando el comité evalúa la solicitud. En esta base, Castigado, Judicial, Refinanciado y Vencido tienen 100 % de mora; Vigente, 0 %. Las gestiones promedian aproximadamente 6,99 para créditos con mora frente a 0,40 sin mora. La asociación es descriptiva; la razón para excluirlas es su disponibilidad **posterior** al momento de predicción.

**Decisión:** Retirar esas dos variables de la salida. Excluir también `monto_aprobado`, `plazo_meses`, `tasa_interes_anual` y `cuota_mensual` de manera conservadora: el diccionario describe condiciones finales y no acredita que sean propuestas ya conocidas antes de la decisión. Esta segunda exclusión es una **incertidumbre de disponibilidad**, no una fuga demostrada por correlación. Conservar `monto_solicitado`. Mantener identificadores y fechas solo como metadatos; en el notebook 03 deben retirarse de `X`, igual que el objetivo. Los históricos de buró y mora previa se conservan bajo la definición del diccionario.

**Justificación:** El objetivo del notebook 01 es apoyar al comité antes de decidir. Un modelo no debe utilizar consecuencias del crédito ni condiciones finales que todavía podría modificar la propia decisión. Si se demuestra que existían propuestas de monto, plazo, tasa y cuota, podrán evaluarse sus versiones previas; no basta con renombrar las finales. La fecha de desembolso puede organizar una evaluación retrospectiva, pero no es predictor de una solicitud aún sin decisión. El código del asesor se conserva suponiendo asignación previa; debe comprobarse y revisarse su estabilidad temporal. No se usa la etiqueta para construir nuevas variables.

### 4.9 Resumen de decisiones

Las cantidades se refieren a 15 395 créditos únicos, salvo que se indique «entrada». Ninguna fila se borra por tener un valor extremo o faltante.

| Problema | Evidencia | Clasificación | Impacto | Decisión | Momento |
|---|---|---|---|---|---|
| Tipos y fechas con dos formatos | 758 ingresos con `S/` y 1 890 fechas D/M/A por columna en la entrada | Inconsistencia de representación | Operaciones incorrectas o pérdida de datos al convertir | Parseo explícito; números e enteros anulables; cero nulos nuevos | Antes del split |
| Copias de créditos | 184 filas idénticas adicionales | Error evidente | Sobreponderación y contaminación entre conjuntos | Quitar solo copias exactas; comprobar unicidad de crédito | Antes del split |
| Clientes repetidos | 3 297 clientes con varios créditos | Repetición válida de la unidad cliente | Dependencia entre observaciones | Conservar créditos; usar ID para auditar solapamiento, fuera de X | Antes del split y al definir split |
| Región, rubro y sexo | 25→9, 16→5 y 4→2 etiquetas; sexo cambia en 2023Q3 | Inconsistencia de categorías | Fragmentación y señal temporal artificial | Mapas semánticos explícitos; equivalencia Lima Metropolitana/Lima revisable | Antes del split |
| Score ausente | 2 377 casos, todos Sin historial | Faltante estructural | Confundir ausencia de historial con bajo score | Mantener nulo/categoría e indicador; mediana solo como entrada numérica del algoritmo | Indicador antes del split; imputación en pipeline |
| Antigüedad ausente | 663, todos en Digital | Faltante problemático asociado a captura | Sesgo por canal si se eliminan filas | Conservar y agregar indicador; mediana ajustada solo en train | Indicador antes del split; imputación en pipeline |
| Ingresos y gastos ausentes | 384 y 615 | Faltante problemático | Pérdida de información sobre capacidad de pago | Conservar nulos e indicadores; mediana de train | Indicador antes del split; imputación en pipeline |
| Educación y teléfono ausentes | 1 265 y 461; educación concentrada en campo | Faltante problemático | Inventar educación o verificación al usar la moda | Conservar indicador; categoría No informado y codificación posterior | Indicador antes del split; imputación/codificación en pipeline |
| Edad 999 | 46 casos | Error evidente | Distorsión del rango y del escalado | Marcar como nulo; no adivinar edad | Antes del split; imputación en pipeline |
| Edad 15 | 30 casos | Valor sospechoso que requiere contexto | Posible registro erróneo o exclusión de titulares válidos | Marcar como nulo bajo supuesto de titular adulto; validar supuesto | Antes del split; imputación en pipeline |
| Antigüedad −1 | 46 casos | Error evidente / posible código de ausencia | Duración inválida | Marcar como nulo | Antes del split; imputación en pipeline |
| Plazo cero | 30 casos | Error evidente | Contrato mensual incoherente | Marcar nulo en la copia corregida; plazo final luego excluido | Antes del split |
| Ingresos/deudas muy altos | 720 ingresos y 1 076 deudas fuera de 1,5 IQR | Valor extremo potencialmente válido | Sensibilidad de algunos modelos | Conservar; no recortar con umbrales de toda la base | Sin recorte; escalado en pipeline |
| Tasa en fracción | 153 tasas entre 0 y 1; 152 cuotas verificables apoyan ×100 | Inconsistencia probable de unidad | Tasa cien veces menor | Multiplicar por 100 en la copia auditada; tasa final luego excluida | Antes del split |
| Fechas invertidas | 76 desembolsos antes de solicitud | Inconsistencia temporal evidente; campo erróneo desconocido | Asignación temporal posiblemente incorrecta | Conservar ambas fechas como metadatos, auditar y pedir fuente; no intercambiarlas | Antes del split: documentar; sensibilidad al evaluar |
| Aprobado mayor a solicitado | 123 casos | Inconsistencia probable | Condiciones posiblemente alteradas | No igualar valores; mantener auditoría y excluir condiciones finales de la salida | Antes del split |
| Negocio más antiguo que el titular | 1 caso con edad 15 | Valor sospechoso que requiere contexto | Corregir el campo equivocado | Tratar edad dudosa; no borrar antigüedad positiva | Antes del split |
| Negocio iniciado antes de los 18 / cuota mayor al ingreso | 49 / 46 alertas | Valor sospechoso que requiere contexto | Borrar negocios familiares o perfiles de riesgo | Conservar; no asumir un error por la alerta | Sin corrección automática |
| Cobranzas y estado al corte | Datos posteriores, fuerte asociación con y | Fuga temporal confirmada | Evaluación artificialmente favorable | Excluir ambas columnas | Antes del split |
| Condiciones finales del contrato | Disponibilidad previa al comité no documentada | Riesgo de fuga por disponibilidad incierta | Usar información decidida después del scoring | Excluir monto aprobado, plazo, tasa y cuota finales | Antes del split |
| Identificadores, fechas y objetivo dentro de X | ID no es atributo; fecha desembolso y y son posteriores | Riesgo de memorización/fuga | Predicción no replicable al evaluar una solicitud | Conservar en CSV para trazabilidad; separar y y quitar metadatos de X | Al construir X/y, antes del split |
| Transformaciones estadísticas | Valores faltantes y escalas heterogéneas | Preparación pendiente | Filtrar información de validation/test a train | Imputar numéricas, categorizar ausencias, escalar y OneHotEncoder con desconocidos ignorados | Pipeline: fit solo en train |

### 4.10 Aplicación de las correcciones

Las correcciones se implementan y ejecutan **en este notebook**, según el alcance de esta entrega. No se importa `src/limpieza.py`; la extracción a `src/` queda para el TF1. La función parte de una copia, conserva el CSV original y devuelve tanto el resultado como una bitácora de cambios. `df_corregido` retiene las columnas finales solo para auditar el antes/después; `df` es la salida destinada al modelado y ya excluye las seis variables de 4.8. Las contradicciones no resolubles se conservan explícitamente como asuntos pendientes, sin inventar reemplazos.

In [10]:
COLS_INDICADOR = ["score_buro", "antiguedad_negocio_meses", "ingreso_mensual_declarado",
                  "gasto_mensual_declarado", "nivel_educativo", "telefono_verificado"]


def limpiar(datos):
    bitacora = [{"accion": "eliminar_copias_exactas", "variable": "fila_completa",
                 "cantidad": int(datos.duplicated().sum())}]
    salida = datos.drop_duplicates(keep="first").copy(deep=True)
    if salida[IDS].isna().any().any() or salida["id_credito"].duplicated().any():
        raise ValueError("Identificadores ausentes o versiones diferentes de un crédito.")
    originales = salida.copy(deep=True)
    salida = convertir_tipos(salida)
    if (originales.notna() & salida.isna()).any().any():
        raise ValueError("La conversión creó nulos no documentados: revisar la entrada.")
    if not salida[TARGET].isin([0, 1]).all():
        raise ValueError("El objetivo debe ser observado y binario; no se imputa.")
    antes_categorias = salida.copy(deep=True)
    salida = unificar_categorias(salida)
    for col in MAPAS:
        bitacora.append({"accion": "unificar_categoria", "variable": col,
                         "cantidad": int(antes_categorias[col].ne(salida[col]).fillna(False).sum())})
    fraccion = salida["tasa_interes_anual"].between(0, 1, inclusive="neither")
    salida.loc[fraccion, "tasa_interes_anual"] = (salida.loc[fraccion, "tasa_interes_anual"] * 100).round(6)
    bitacora.append({"accion": "convertir_fraccion_a_porcentaje", "variable": "tasa_interes_anual",
                     "cantidad": int(fraccion.sum())})
    for col, mascara in mascaras_invalidos(salida).items():
        if mascara.any():
            bitacora.append({"accion": "marcar_invalido_como_nulo", "variable": col,
                             "cantidad": int(mascara.sum())})
            salida.loc[mascara, col] = pd.NA if col in ENTEROS else np.nan
    for col in COLS_INDICADOR:
        salida[f"{col}_faltante"] = salida[col].isna().astype("int64")
    corregido = salida.copy(deep=True)
    for col in EXCLUIR_SALIDA:
        bitacora.append({"accion": "excluir_columna", "variable": col, "cantidad": len(salida)})
    salida = salida.drop(columns=EXCLUIR_SALIDA).reset_index(drop=True)
    return salida, corregido.reset_index(drop=True), pd.DataFrame(bitacora)


df, df_corregido, bitacora = limpiar(df_raw)
display(bitacora)
print(f"Resultado: {df.shape[0]:,} créditos; {df.shape[1]} columnas")

,accion,variable,cantidad
0,eliminar_copias_exactas,fila_completa,184
1,unificar_categoria,region,2155
2,unificar_categoria,rubro,1539
3,unificar_categoria,sexo,6629
4,convertir_fraccion_a_porcentaje,tasa_interes_anual,153
5,marcar_invalido_como_nulo,edad,76
6,marcar_invalido_como_nulo,antiguedad_negocio_meses,46
7,marcar_invalido_como_nulo,plazo_meses,30
8,excluir_columna,num_gestiones_cobranza,15395
9,excluir_columna,estado_credito_corte,15395


Resultado: 15,395 créditos; 35 columnas


In [11]:
def resumen_base(datos):
    return {
        "filas": len(datos), "columnas": datos.shape[1],
        "creditos_distintos": datos["id_credito"].nunique(),
        "clientes_distintos": datos["id_cliente"].nunique(),
        "copias_exactas": int(datos.duplicated().sum()),
        "celdas_faltantes": int(datos.isna().sum().sum()),
        "filas_con_faltantes": int(datos.isna().any(axis=1).sum()),
        "creditos_con_mora": int(datos[TARGET].sum()),
        "porcentaje_mora": datos[TARGET].mean() * 100,
    }


display(pd.DataFrame({"entrada": resumen_base(df_raw), "solo_sin_copias": resumen_base(sin_duplicados),
                      "corregido_con_auditoria": resumen_base(df_corregido),
                      "salida": resumen_base(df)}).T)
comparacion = pd.DataFrame({
    "tipo_entrada": df_raw.dtypes.astype(str), "tipo_salida": df.dtypes.astype(str),
    "nulos_entrada": df_raw.isna().sum(), "nulos_sin_copias": sin_duplicados.isna().sum(),
    "nulos_salida": df.isna().sum(),
    "pct_nulos_entrada": df_raw.isna().mean() * 100,
    "pct_nulos_salida": df.isna().mean() * 100,
})
comparacion["situacion"] = ["excluida" if c not in df else "agregada" if c not in df_raw else "conservada"
                            for c in comparacion.index]
display(comparacion)
display(pd.DataFrame({"categorias_antes": df_base[list(MAPAS)].nunique(),
                      "categorias_salida": df[list(MAPAS)].nunique()}))
display(pd.DataFrame({"invalidos_antes": {c: int(m.sum()) for c, m in invalidos.items()},
                      "invalidos_despues": {c: int(m.sum()) for c, m in mascaras_invalidos(df_corregido).items()}}))
display(pd.DataFrame({"alertas_antes": reglas_antes.sum(),
                      "alertas_despues_en_copia_auditada": revisar_reglas(df_corregido).sum()}))
display(pd.DataFrame({"tasa_antes": df_diag["tasa_interes_anual"].describe(),
                      "tasa_corregida_solo_auditoria": df_corregido["tasa_interes_anual"].describe()}))

assert len(df) == len(df_raw) - int(df_raw.duplicated().sum())
assert df["id_credito"].is_unique
assert df[TARGET].notna().all() and df[TARGET].isin([0, 1]).all()
assert not set(EXCLUIR_SALIDA).intersection(df.columns)
assert not any(m.any() for m in mascaras_invalidos(df_corregido).values())
assert not df_corregido["tasa_interes_anual"].between(0, 1, inclusive="neither").any()
assert df[TARGET].sum() == sin_duplicados[TARGET].sum()
for col in COLS_INDICADOR:
    assert df[f"{col}_faltante"].equals(df[col].isna().astype("int64"))
pd.testing.assert_frame_equal(df_raw, pd.read_csv(RAIZ / "data/raw/creditos_mype.csv"))
print("Verificaciones de integridad superadas; la fuente permanece intacta.")

,filas,columnas,creditos_distintos,clientes_distintos,copias_exactas,celdas_faltantes,filas_con_faltantes,creditos_con_mora,porcentaje_mora
entrada,"15,579.0000",35.0000,"15,395.0000","11,500.0000",184.0000,"5,856.0000","5,132.0000","1,692.0000",10.8608
solo_sin_copias,"15,395.0000",35.0000,"15,395.0000","11,500.0000",0.0000,"5,765.0000","5,060.0000","1,679.0000",10.9061
corregido_con_auditoria,"15,395.0000",41.0000,"15,395.0000","11,500.0000",0.0000,"5,917.0000","5,170.0000","1,679.0000",10.9061
salida,"15,395.0000",35.0000,"15,395.0000","11,500.0000",0.0000,"5,887.0000","5,150.0000","1,679.0000",10.9061


,tipo_entrada,tipo_salida,nulos_entrada,nulos_sin_copias,nulos_salida,pct_nulos_entrada,pct_nulos_salida,situacion
antiguedad_negocio_meses,float64,Int64,671.0000,663.0000,709.0000,4.3071,4.6054,conservada
antiguedad_negocio_meses_faltante,NaN,int64,NaN,NaN,0.0000,NaN,0.0000,agregada
calificacion_sbs,str,string,0.0000,0.0000,0.0000,0.0000,0.0000,conservada
canal_captacion,str,string,0.0000,0.0000,0.0000,0.0000,0.0000,conservada
cliente_recurrente,str,string,0.0000,0.0000,0.0000,0.0000,0.0000,conservada
codigo_asesor,str,string,0.0000,0.0000,0.0000,0.0000,0.0000,conservada
cuota_mensual,float64,NaN,0.0000,0.0000,NaN,0.0000,NaN,excluida
deuda_total_sistema,float64,float64,0.0000,0.0000,0.0000,0.0000,0.0000,conservada
edad,int64,Int64,0.0000,0.0000,76.0000,0.0000,0.4937,conservada
estado_civil,str,string,0.0000,0.0000,0.0000,0.0000,0.0000,conservada


,categorias_antes,categorias_salida
region,25,9
rubro,16,5
sexo,4,2


,invalidos_antes,invalidos_despues
edad,76,0
antiguedad_negocio_meses,46,0
plazo_meses,30,0
score_buro,0,0
tasa_interes_anual,0,0
ingreso_mensual_declarado,0,0
monto_solicitado,0,0
monto_aprobado,0,0
cuota_mensual,0,0
gasto_mensual_declarado,0,0


,alertas_antes,alertas_despues_en_copia_auditada
desembolso_anterior_a_solicitud,76,76
desembolso_fuera_2022_2024,0,0
sin_seis_meses_al_corte,0,0
objetivo_fuera_0_1_o_ausente,0,0
aprobado_mayor_solicitado,123,123
negocio_mayor_edad_registrada,1,0
negocio_previo_mayoria_edad_con_edad_valida,49,49
recurrente_incompatible_con_creditos_previos,0,0
cero_entidades_con_deuda,0,0
entidades_con_deuda_cero,0,0


,tasa_antes,tasa_corregida_solo_auditoria
count,"15,395.0000","15,395.0000"
mean,57.6189,58.1914
std,9.7518,7.9467
min,0.2851,22.0000
25%,53.0500,53.2300
50%,58.4200,58.5400
75%,63.5050,63.5600
max,85.6400,85.6400


Verificaciones de integridad superadas; la fuente permanece intacta.


**Interpretación:** Se pasa de **15 579 filas a 15 395 créditos únicos**, conservando los 11 500 clientes. Solo se eliminan las 184 copias exactas: ninguna fila se descarta por edad, monto extremo, faltantes o alertas de negocio. La mora pasa de **10,8608 % a 10,9061 %** (1 692 a 1 679 registros positivos), un aumento de 0,0454 puntos porcentuales explicado únicamente por la deduplicación. Las correcciones no cambian ninguna etiqueta de los créditos únicos.

Las categorías de región/rubro/sexo quedan en 9/5/2 y la conversión de tipos no introduce nulos. Tras quitar copias, marcar los errores agrega 76 nulos de edad, 46 de antigüedad y 30 de plazo en la copia auditada; el plazo se elimina después por disponibilidad temporal incierta. La antigüedad queda con 709 faltantes. El CSV final tiene **35 columnas**: 35 originales menos 6 excluidas más 6 indicadores. Que mantenga el mismo número de columnas no significa que mantenga las mismas variables. La comparación incluye una base solo deduplicada para distinguir reducción de nulos por eliminación de copias y aumento por errores marcados.

Las 76 contradicciones de fechas y las 123 de montos no han sido reparadas: permanecen en los datos de auditoría; las fechas se conservan como metadatos y los montos finales se excluyen de la salida. La alerta de negocio más antiguo que la edad deja de ser verificable al marcar la edad dudosa, **no queda demostrada como resuelta**. Se conservan los valores monetarios extremos. Un archivo limpio para esta etapa puede contener nulos y asuntos documentados pendientes de validación.

**Uso en el notebook 03:** reemplazar su limpieza temporal por la lectura del CSV con `parse_dates=["fecha_solicitud", "fecha_desembolso"]`; el formato CSV no conserva tipos de fecha ni `Int64`. Separar `y = mora_30d_6m` y retirar identificadores y ambas fechas de `X`. Hacer el split antes de ajustar medianas, escalado y codificación; representar las categorías ausentes como `No informado` y conservar los seis indicadores. Revisar las fechas contradictorias al elegir el split. No se modifica el notebook 03 en esta entrega ni se entrena un modelo aquí.

In [12]:
RUTA_SALIDA = RAIZ / "data/processed/creditos_limpio.csv"
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_SALIDA, index=False, encoding="utf-8", date_format="%Y-%m-%d")

recargado = pd.read_csv(RUTA_SALIDA, parse_dates=FECHAS)
assert recargado.shape == df.shape
assert recargado.columns.tolist() == df.columns.tolist()
assert recargado["id_credito"].is_unique
assert recargado[TARGET].tolist() == df[TARGET].tolist()
assert recargado.isna().sum().to_dict() == df.isna().sum().to_dict()
for col in FECHAS:
    assert recargado[col].eq(df[col]).all()
print(f"Guardado y verificado: {RUTA_SALIDA.relative_to(RAIZ)}")
print(f"{len(recargado):,} filas, {recargado.shape[1]} columnas; faltantes conservados para el pipeline.")

Guardado y verificado: data\processed\creditos_limpio.csv
15,395 filas, 35 columnas; faltantes conservados para el pipeline.


### 4.11 Incertidumbre y riesgos de las decisiones

- **Decisiones con mayor incertidumbre:** La disponibilidad de las condiciones finales antes del comité es la principal: excluirlas puede ser demasiado conservador, pero incluirlas sin su versión previa puede producir fuga. También son revisables la edad mínima asumida, la equivalencia Lima Metropolitana/Lima, el tratamiento del score ausente y la conservación de extremos. La conversión de tasas tiene evidencia fuerte en las cuotas, aunque no prueba por sí sola el origen del error. Las 76 contradicciones temporales y las 123 de montos siguen sin una corrección identificable. La ausencia de antigüedad por canal sugiere un mecanismo de captura; no demuestra que sea un campo no aplicable.

- **Qué se perdería o sesgaría si fueran incorrectas:** Quitar condiciones realmente conocidas reduciría capacidad predictiva; incluir condiciones decididas después inflaría la evaluación y haría imposible replicarla en producción. Marcar edades de 15 como nulas perdería información si esos titulares fueran válidos. Unificar Lima podría ocultar diferencias geográficas. Multiplicar una tasa realmente inferior a 1 % por 100 sobreestimaría el costo del crédito. Imputar el score estructural con una mediana puede ocultar diferencias entre personas con y sin historial; el indicador y la categoría reducen ese riesgo, pero no lo eliminan. Mantener ingresos extremos erróneos afectaría el ajuste, mientras recortar ingresos legítimos invisibilizaría negocios más grandes. Conservar fechas posiblemente erradas puede ubicar créditos en un conjunto temporal incorrecto; no se atribuye validez a una fecha solo porque sea interpretable. Los indicadores de ausencia también podrían perder utilidad si cambia el proceso de captura.

- **Qué se consultaría a alguien del área de créditos:** ¿El scoring se aplica antes de fijar condiciones o con una propuesta ya registrada? ¿Existen versiones y sellos de tiempo de monto, plazo, tasa y cuota? ¿Cuál es el criterio real de edad del titular? ¿−1 significa ausencia de antigüedad? ¿Lima y Lima Metropolitana identifican el mismo nivel territorial? ¿Las tasas se almacenaron como fracción en algún sistema? ¿Las cuotas incluyen seguros o comisiones? ¿Qué fuente permite corregir las fechas invertidas y los montos renegociados? ¿La antigüedad digital no se pregunta o se perdió al extraerla? ¿Sin historial implica siempre ausencia de score? ¿Los antecedentes de buró y el asesor corresponden efectivamente a la evaluación original?

- **Cómo evaluar la sensibilidad sin contaminar test:** En el notebook 03, comparar en train/validation la inclusión de versiones previas verificadas de las condiciones, una transformación monetaria justificada y ajustada solo en train, y la evaluación con/sin los 76 créditos de fechas contradictorias. Examinar métricas por canal y por historial de buró, además del resultado global. No escoger una limpieza por su rendimiento en test ni modificar reglas según la mora de una fila. Reservar test para la evaluación final; antes de usar la base en decisiones reales, resolver la trazabilidad con la fuente.